# Bayesian Optimisation Capstone — Week 1 Queries

This notebook covers the full pipeline for Week 1:
- Loading and exploring the initial data for all 8 functions
- Fitting a Gaussian Process (GP) surrogate model to each function
- Using the Upper Confidence Bound (UCB) acquisition function to select the next query point
- Formatting and printing the final submission strings

**Method:** GP + UCB with κ = 2.576 and 50,000 random candidates per function

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# Reproducibility
np.random.seed(42)

print('All libraries loaded successfully.')

## 2. Load Initial Data

Each function has an inputs `.npy` file (shape: n_samples x n_dims) and an outputs `.npy` file (shape: n_samples,).

Update `DATA_PATH` to point to the folder containing your `.npy` files.

In [ ]:
DATA_PATH = './data/'  # <-- update this path if needed

functions = {}
for i in range(1, 9):
    X = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    functions[i] = {'X': X, 'y': y}

print(f'{'Function':<12} {'Dims':<6} {'N pts':<8} {'Y min':<12} {'Y max':<12} {'Best Y'}')
print('-' * 62)
for i, d in functions.items():
    X, y = d['X'], d['y']
    print(f'Function {i:<3} {X.shape[1]:<6} {len(y):<8} {y.min():<12.4f} {y.max():<12.4f} {y.max():.6f}')

## 3. Exploratory Data Analysis

Quick visual summaries. For the two 2D functions (F1 and F2) we can plot the known points directly. For higher-dimensional functions we plot the output distribution.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (func_id, d) in enumerate(functions.items()):
    X, y = d['X'], d['y']
    ax = axes[i]
    
    if X.shape[1] == 2:
        # Scatter plot for 2D functions — colour by output value
        sc = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=80, edgecolors='k', linewidths=0.5)
        plt.colorbar(sc, ax=ax, label='y')
        ax.set_xlabel('x1')
        ax.set_ylabel('x2')
    else:
        # Bar chart of output values for higher-dimensional functions
        ax.bar(range(len(y)), sorted(y), color='steelblue', edgecolor='navy', linewidth=0.4)
        ax.axhline(y.max(), color='red', linestyle='--', linewidth=1.2, label=f'Best: {y.max():.3f}')
        ax.legend(fontsize=8)
        ax.set_xlabel('Sample index (sorted)')
        ax.set_ylabel('Output y')
    
    ax.set_title(f'Function {func_id} ({X.shape[1]}D, n={len(y)})', fontsize=11, fontweight='bold')

plt.suptitle('Initial Data Overview — All 8 Functions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as eda_overview.png')

## 4. GP + UCB Acquisition Function

### How it works

1. **Gaussian Process (GP):** A probabilistic model fitted to the observed (X, y) data. For any new candidate point x, the GP returns:
   - **μ(x):** predicted mean output
   - **σ(x):** predicted uncertainty (standard deviation)

2. **UCB acquisition function:** Scores each candidate as:
   
   `UCB(x) = μ(x) + κ * σ(x)`
   
   - Higher κ → more exploration (prefer uncertain regions)
   - Lower κ → more exploitation (prefer high predicted mean)
   - We use **κ = 2.576** in Round 1 (high uncertainty, early stage)

3. **Candidate search:** 50,000 random points are drawn uniformly from [0, 1]^d and scored. The highest UCB point is selected.

In [ ]:
def fit_gp(X, y):
    """
    Fit a Gaussian Process regressor to the data.
    Uses a Matern 5/2 kernel, which is suitable for functions
    that are twice differentiable — a common assumption for
    black-box optimisation problems.
    """
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,  # Avoid poor local optima in kernel fitting
        normalize_y=True          # Helps with functions at very different scales
    )
    gp.fit(X, y)
    return gp


def ucb_acquisition(X_candidates, gp, kappa=2.576):
    """
    Compute UCB acquisition scores for a set of candidate points.
    
    Parameters
    ----------
    X_candidates : array of shape (n_candidates, n_dims)
    gp           : fitted GaussianProcessRegressor
    kappa        : exploration-exploitation trade-off parameter
    
    Returns
    -------
    ucb_scores : array of shape (n_candidates,)
    """
    mu, sigma = gp.predict(X_candidates, return_std=True)
    return mu + kappa * sigma


def suggest_next_query(X, y, n_dims, n_candidates=50000, kappa=2.576, seed=42):
    """
    Full pipeline: fit GP, score candidates with UCB, return best point.
    
    Returns
    -------
    best_x     : array of shape (n_dims,) — the recommended query point
    mu         : predicted mean at best_x
    sigma      : predicted std at best_x
    ucb_score  : UCB score at best_x
    """
    rng = np.random.default_rng(seed)
    gp = fit_gp(X, y)
    
    # Sample candidates uniformly from the unit hypercube
    candidates = rng.uniform(0, 1, size=(n_candidates, n_dims))
    
    # Score all candidates with UCB
    ucb_scores = ucb_acquisition(candidates, gp, kappa)
    
    # Select the best candidate
    best_idx = np.argmax(ucb_scores)
    best_x = candidates[best_idx]
    
    mu, sigma = gp.predict(best_x.reshape(1, -1), return_std=True)
    
    return best_x, float(mu[0]), float(sigma[0]), float(ucb_scores[best_idx])


def format_query(x):
    """Format a query vector as the required submission string."""
    return '-'.join([f'{v:.6f}' for v in x])


print('Functions defined. Ready to run queries.')

## 5. Run Queries for All 8 Functions

In [ ]:
results = {}

for func_id, d in functions.items():
    X, y = d['X'], d['y']
    n_dims = X.shape[1]
    
    best_x, mu, sigma, ucb = suggest_next_query(X, y, n_dims)
    query_str = format_query(best_x)
    
    results[func_id] = {
        'best_x': best_x,
        'query_string': query_str,
        'predicted_mu': mu,
        'predicted_sigma': sigma,
        'ucb_score': ucb,
        'current_best_y': y.max(),
        'current_best_x': X[np.argmax(y)]
    }
    
    print(f'Function {func_id} ({n_dims}D)')
    print(f'  Current best y : {y.max():.6f}')
    print(f'  Query          : {query_str}')
    print(f'  GP mu          : {mu:.4f}  |  sigma: {sigma:.4f}  |  UCB: {ucb:.4f}')
    print()

## 6. Formatted Submission Strings

Copy these directly into the capstone portal.

In [ ]:
print('=' * 55)
print('WEEK 1 SUBMISSION STRINGS')
print('=' * 55)
for func_id, r in results.items():
    print(f'Function {func_id}: {r["query_string"]}')
print('=' * 55)

## 7. UCB Landscape Visualisation (Functions 1 and 2 only)

For the two 2D functions, we can visualise the GP mean, uncertainty and UCB surface across the full [0,1]^2 space. This shows exactly why the GP recommended each query point.

In [ ]:
def plot_2d_gp(func_id, X, y, query_x, kappa=2.576, resolution=80):
    """Plot GP mean, sigma, and UCB surface for a 2D function."""
    
    # Build a grid
    x1 = np.linspace(0, 1, resolution)
    x2 = np.linspace(0, 1, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    
    # Fit GP and predict
    gp = fit_gp(X, y)
    mu, sigma = gp.predict(grid, return_std=True)
    ucb = mu + kappa * sigma
    
    mu_grid    = mu.reshape(resolution, resolution)
    sigma_grid = sigma.reshape(resolution, resolution)
    ucb_grid   = ucb.reshape(resolution, resolution)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    panels = [
        (mu_grid,    'GP Mean (mu)',        'RdYlGn'),
        (sigma_grid, 'GP Uncertainty (sigma)', 'Blues'),
        (ucb_grid,   f'UCB Score (kappa={kappa})', 'plasma')
    ]
    
    for ax, (data, title, cmap) in zip(axes, panels):
        im = ax.contourf(X1, X2, data, levels=30, cmap=cmap)
        plt.colorbar(im, ax=ax)
        
        # Observed points
        ax.scatter(X[:, 0], X[:, 1], c='white', s=60, edgecolors='black',
                   linewidths=1.2, zorder=5, label='Observed')
        
        # Recommended query point
        ax.scatter(query_x[0], query_x[1], c='red', s=120, marker='*',
                   edgecolors='black', linewidths=0.8, zorder=6, label='Query')
        
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('x1')
        ax.set_ylabel('x2')
        ax.legend(fontsize=8)
    
    plt.suptitle(f'Function {func_id} — GP Surfaces and UCB Query', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'function_{func_id}_gp_surfaces.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: function_{func_id}_gp_surfaces.png')


# Plot for Functions 1 and 2 (both 2D)
for func_id in [1, 2]:
    d = functions[func_id]
    r = results[func_id]
    plot_2d_gp(func_id, d['X'], d['y'], r['best_x'])

## 8. Summary Table of All Queries

In [ ]:
import pandas as pd

rows = []
for func_id, r in results.items():
    rows.append({
        'Function': f'F{func_id}',
        'Dims': functions[func_id]['X'].shape[1],
        'N pts': len(functions[func_id]['y']),
        'Current Best Y': round(r['current_best_y'], 4),
        'GP Pred Mean': round(r['predicted_mu'], 4),
        'GP Uncertainty': round(r['predicted_sigma'], 4),
        'UCB Score': round(r['ucb_score'], 4),
        'Query String': r['query_string']
    })

df = pd.DataFrame(rows)
df.set_index('Function', inplace=True)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)
print(df.to_string())

## 9. How to Update

Update the data like this:

```python
# Example: received y = 0.42 for Function 2
new_x = np.array([0.813958, 0.968718])  # the query point you submitted
new_y = 0.42                             # the output value you received back

# Append to the existing data
functions[2]['X'] = np.vstack([functions[2]['X'], new_x])
functions[2]['y'] = np.append(functions[2]['y'], new_y)

# Then re-run Section 5 to generate new queries
# Consider reducing kappa slightly, e.g. kappa=2.0, for more exploitation
```

### Kappa schedule
| Round | Kappa | Rationale |
|-------|-------|-----------|
| Week 1 | 2.576 | High exploration — limited data |
| Week 2 | 2.0   | Slightly more exploitation |
| Week 3 | 1.5   | Balanced |
| Week 4+ | 1.0  | Mostly exploitation — converging |